In [22]:
import pandas as pd

In [23]:
#these are words to indicate whether something is broken or not working
urgent_keywords = [
    "error", "fail", "crash", "broken", "not working", "exception",
    "bug", "fix", "security", "vulnerability", "memory leak",
    "permission denied", "access denied", "corrupt", "block",
    "issue", "problem"
]

#words that inidcate a question or request for help
help_keywords = [
    "how to", "how do i", "how can i", "difference between",
    "best practice", "performance", "optimize", "upgrade", "migrate",
    "convert", "deprecated", "alternative", "what is", "what does",
    "why is", "why does", "when to use"
]

In [24]:
#map labels to their priority and unknown priority goes to medium
#bug is always high priority, the others will get a keyword check to determine a bump up
labels_map = {
    "BUG": "High",
    "ENH": "Medium",
    "RFC": "Medium",
    "MAINT": "Medium",
    "BLD": "Medium",
    "DEP": "Medium",
    "CI": "Medium",
    "DOC": "Low",
    "TST": "Low",
    "BENCH": "Low",
    "DEV": "Low",
    "TASK": "Low",
    "QUERY": "Low",
    "TYP": "Low",
}
default_priority = "Medium"

In [25]:
#rank mapping so we can bump priority up by one level
PRIORITY_RANK = {"Low": 0, "Medium": 1, "High": 2}
RANK_TO_LABEL = {0: "Low", 1: "Medium", 2: "High"}


In [26]:
#loading kaggle train and test data
kaggle_train = pd.read_csv("/Users/davidmoriarty/Downloads/university_query_train.csv")
kaggle_test = pd.read_csv("/Users/davidmoriarty/Downloads/university_query_test.csv")

print(kaggle_train.shape)
print(kaggle_test.shape)

(5000, 5)
(1000, 5)


In [27]:
#loading github issues
df_github = pd.read_csv("/Users/davidmoriarty/Documents/github_issues_raw.csv")
print(df_github.shape)
print(df_github.head())

(1000, 2)
         source                                           raw_text
0  github_scipy  BUG: `_batched_linalg` wrapper for `linalg.sol...
1  github_scipy  ENH: interpolate.RBFInterpolator: add Matérn k...
2  github_scipy  RFC: declaring netcdf IO functions in `scipy.i...
3  github_scipy  DOC: return _RichResult is a bit strange with ...
4  github_scipy  DOC: `elementwise.find_root`: transforming a c...


In [ ]:
#this is to get priority from stackoverflow score
def get_stack_priority(row):
    lowercase = str(row["raw_text"]).lower()
    score = row["Score"]

    # first look: keyword classification
    if any(keyword in lowercase for keyword in urgent_keywords):
        return "High"
    elif any(keyword in lowercase for keyword in help_keywords):
        return "Medium"

    # look 2: high score questions that missed keywords are probably still important
    if score >= 3000:
        return "Medium"
    return "Low"

#loading stackoverflow data and cleaning it to keep the data fields we need
df_stack = pd.read_csv("/Users/davidmoriarty/Downloads/QueryResults.csv")
print(df_stack.shape)
print(df_stack.head())

stack_clean = pd.DataFrame({
    "source": df_stack["source"],
    "raw_text": df_stack["raw_text"],
    "priority_label": df_stack.apply(get_stack_priority, axis=1)
})

print(stack_clean.shape)
print(stack_clean["priority_label"].value_counts())
stack_clean.head()

(2000, 3)
priority_label
Medium    1164
Low        748
High        88
Name: count, dtype: int64


,source,raw_text,priority_label
0,stackoverflow,Why is processing a sorted array faster than p...,Medium
1,stackoverflow,How do I undo the most recent local commits in...,Medium
2,stackoverflow,How do I delete a Git branch locally and remot...,Medium
3,stackoverflow,What is the difference between 'git pull' and ...,Medium
4,stackoverflow,"What does the ""yield"" keyword do in Python?",Medium


In [29]:
# cleaning kaggle data to keep the data fields we need 
kaggle_train_clean = pd.DataFrame({
    "source": "kaggle",
    "raw_text": kaggle_train["Student_Query"],
    "priority_label": kaggle_train["Priority_Label"]
})

kaggle_test_clean = pd.DataFrame({
    "source": "kaggle",
    "raw_text": kaggle_test["Student_Query"],
    "priority_label": kaggle_test["Priority_Label"]
})

print(kaggle_train_clean.shape)
kaggle_train_clean.head()

(5000, 3)


,source,raw_text,priority_label
0,kaggle,How to join student clubs?,Low
1,kaggle,My admit card has incorrect details.,High
2,kaggle,How to reset my university portal password?,Medium
3,kaggle,My exam form is not submitted and tomorrow is ...,High
4,kaggle,I cannot download my hall ticket for tomorrow'...,High


In [30]:
# function to get priority from the title prefix and check keywords
def get_priority(title):
    label = title.split(":")[0].strip()
    starting_priority = labels_map.get(label, default_priority)
    body = title.lower()
    #bug is always high, skip the keyword check
    if label == "BUG":
        return "High"
    #for ambiguous labels check if the description has urgency keywords
    if any(keyword in body for keyword in urgent_keywords):
        bump_up = min(PRIORITY_RANK[starting_priority] + 1, 2)
        return RANK_TO_LABEL[bump_up]
    return starting_priority


# cleaning github data to keep the three schema columns
github_clean = pd.DataFrame({
    "source": df_github["source"],
    "raw_text": df_github["raw_text"],
    "priority_label": df_github["raw_text"].apply(get_priority)
})

print(github_clean.shape)
github_clean.head()

(1000, 3)


,source,raw_text,priority_label
0,github_scipy,BUG: `_batched_linalg` wrapper for `linalg.sol...,High
1,github_scipy,ENH: interpolate.RBFInterpolator: add Matérn k...,Medium
2,github_scipy,RFC: declaring netcdf IO functions in `scipy.i...,Medium
3,github_scipy,DOC: return _RichResult is a bit strange with ...,Low
4,github_scipy,DOC: `elementwise.find_root`: transforming a c...,Low


In [31]:
#now combine all three datasets into one dataset
combined_queries = pd.concat([kaggle_train_clean, kaggle_test_clean, github_clean, stack_clean], ignore_index=True)

print(combined_queries.shape)
print(combined_queries["source"].value_counts())
print(combined_queries["priority_label"].value_counts())
combined_queries.to_csv("/Users/davidmoriarty/Downloads/completed_preprocessed.csv", index=False)
print("saved to completed_preprocessed.csv")

(9000, 3)
source
kaggle           6000
stackoverflow    2000
github_scipy     1000
Name: count, dtype: int64
priority_label
Medium    3381
Low       2973
High      2646
Name: count, dtype: int64
saved to completed_preprocessed.csv
